Stanford Online High School OM065: Short Project #1
Correlation & Simple Linear Regression
Applied to a Virtual Used Car Lot
SITUATION:
Suppose that you are interested in purchasing a used car. How much should you expect to pay?
Obviously the price will depend on the type of car you get (the model) and how much it’s been
used. For this project you will investigate how the price might depend on the age of the car (in
years with this year's models =1 year old). While collecting data, you’ll also want to find and
record the mileage of the car. Code both the prices and mileage in thousands with one decimal
place. Thus a 2003 Honda Civic with 75,670 miles that’s selling for $5,685 would be coded as
age=15, miles=75.7, price=5.7. Please put the data in your dataset with the variable names age,
miles, and price in that order for consistency.
DATA SOURCE:
To provide a convenient location to sample lots of cars, we'll do our ―shopping‖ on the Internet.
You may try one of the sites listed below or find one of your own. You should focus on a single
car model (for example, a Dodge Caravan) when you search for price listings and try to choose a
car model that’s been around for a while (so you get some variety in ages). Be careful if the
results of your search are displayed in order. You should find prices for as random a sample as
feasible with at least 25 cars. Be sure that you are getting prices for actual cars—not ―blue book.‖
theoretical prices. Also, try to get a reasonable range of years.
Web sites to try: autobytel.com, autotrader.com, or, cars.com.
REPORT:
1. Start with an introduction that describes your data, source, and procedure for choosing the
cars to include.
2. Use R to directly compute each of the following summary statistics. Include both the values
and the R commands you use to find them.
 ̅, sx
, ̅, sy,
, SSX, SSXY, SSY (same as SSTotal), SSModel, SSE
3. Show how to calculate the least squares regression line that best fits your data using the
values generated in (2)—you can confirm the result with R. Interpret (in context) what the slope
estimate tells you about prices and ages of your used car model. Explain why the sign
(positive/negative) make sense.
4. Again, using the values in (2) above, show how to estimate the standard deviation of the
error term for your model.
5. Produce scatterplot of the relationship with the regression line drawn on it.
6. Produce residual plots and comment on how well your data appear to fit the conditions for a
simple linear model. Don’t worry about doing transformations if there are problems with the
conditions.
7. Find the car in your sample with the largest residual (in magnitude—positive or negative).
For that car, use R to find its studentized residual, leverage, and Cook’s distance. Would any of
these values be considered unusual? Specify the criteria you use for each measure. 
8. Compute and interpret a 90% confidence interval for the slope of your model (show
calculation).
9. Compute the value of r
2
two ways—as the square of the correlation and using the partitioned
sums of squares from the ANOVA. Write a sentence that interprets the result as a percentage in
context.
10. Test the strength of the linear relationship between your variables using each of the three
methods discussed in class. Show hypotheses and the details for calculating the test statistic in
each case. Indicate the reference distribution (t or F including degrees of freedom) and use
technology to get any p-value(s). One conclusion should suffice for all the tests.
 test for correlation
 test for slope
 ANOVA for regression
11. Choose a particular value of age for which you are interested in predicting the price of a car.
Show how to calculate each of the following quantities! You may confirm the values with R.
Write sentences that carefully interpret each of the intervals (in terms of car prices) and show
the distinction between them.
 predicted value for Y
 90% confidence interval for μY.
 90% prediction interval for an individual Y.
12. According to your model, is there an age at which the car should be free? If so, find this age
and comment on what the ―free car‖ phenomenon says about the appropriateness of your model.
13. Write a conclusion. Discuss your overall impressions of the linear model for describing your
data. Point out any unusual data values, interesting phenomena, or obvious departures from
regression assumptions.
Technology notes:
Note #1: Enter the data using a package (e.g., Excel, Fathom, or Minitab) that will let you save it
as a comma separated (.csv) file. The first row should be the names of the three variables age,
miles, price. You will need to upload that file to a folder in your RStudio workspace and then
import the dataset.
Note #2: Your project report should be word-processed, with graphs, R commands, and ouput
embedded along with interpretations. You do not need to typeset calculations—feel free to leave
some space and write them in by hand.
Stanford Online High School OM065: Short Project #1 

In [ ]:
logf <- function(name) file.path("logs", paste0(name, ".txt"))
run_chunk <- function(name, expr) {
  con <- file(logf(name), open = "wt")
  sink(con, split = FALSE)
  tryCatch({ expr() }, error = function(e) cat("Error:", conditionMessage(e), "\n"))
  sink(); close(con)
}

cars <- read.csv("cars.csv")
n <- nrow(cars)

## ---------------------------------------------------------------
## 2. Summary statistics
## ---------------------------------------------------------------
run_chunk("02_summary_stats", function() {
  cat("n =", n, "\n\n")

  xbar <- mean(cars$age); sx <- sd(cars$age)
  ybar <- mean(cars$price); sy <- sd(cars$price)
  r <- cor(cars$age, cars$price)

  cat("mean(age)      xbar =", xbar, "\n")
  cat("sd(age)        sx   =", sx, "\n")
  cat("mean(price)    ybar =", ybar, "\n")
  cat("sd(price)      sy   =", sy, "\n")
  cat("cor(age,price) r    =", r, "\n\n")

  SSX <- sum((cars$age - xbar)^2)
  SSY <- sum((cars$price - ybar)^2)
  SSXY <- sum((cars$age - xbar) * (cars$price - ybar))

  cat("SSX  = sum((age-xbar)^2)         =", SSX, "\n")
  cat("SSXY = sum((age-xbar)(price-ybar))=", SSXY, "\n")
  cat("SSY  = sum((price-ybar)^2) (=SSTotal) =", SSY, "\n\n")

  b1 <- SSXY / SSX
  b0 <- ybar - b1 * xbar
  cat("slope     b1 = SSXY/SSX =", b1, "\n")
  cat("intercept b0 = ybar - b1*xbar =", b0, "\n\n")

  SSModel <- b1^2 * SSX     # = b1*SSXY
  SSE <- SSY - SSModel

  cat("SSModel = b1*SSXY =", SSModel, "\n")
  cat("SSE = SSY - SSModel =", SSE, "\n")

  # store for later chunks
  saveRDS(list(xbar=xbar, sx=sx, ybar=ybar, sy=sy, r=r,
               SSX=SSX, SSY=SSY, SSXY=SSXY, b0=b0, b1=b1,
               SSModel=SSModel, SSE=SSE, n=n), "stats.rds")
})

## ---------------------------------------------------------------
## 3. Regression line (confirm with lm())
## ---------------------------------------------------------------
run_chunk("03_lm_fit", function() {
  fit <- lm(price ~ age, data = cars)
  print(summary(fit))
  saveRDS(fit, "fit.rds")
})

## ---------------------------------------------------------------
## 4. Estimate of error standard deviation
## ---------------------------------------------------------------
run_chunk("04_se", function() {
  s <- readRDS("stats.rds")
  se <- sqrt(s$SSE / (s$n - 2))
  cat("s = sqrt(SSE/(n-2)) =", se, "\n")
  fit <- readRDS("fit.rds")
  cat("\nConfirm with summary(fit)$sigma:\n")
  print(summary(fit)$sigma)
})

## ---------------------------------------------------------------
## 5. Scatterplot with regression line
## ---------------------------------------------------------------
fit <- readRDS("fit.rds")
png("figs/scatter_regline.png", width = 700, height = 550)
plot(cars$age, cars$price, xlab = "Age (years)", ylab = "Price ($1000s)",
     main = "Used Honda Civic Price vs. Age", pch = 16, col = "steelblue")
abline(fit, col = "red", lwd = 2)
dev.off()

## ---------------------------------------------------------------
## 6. Residual plots
## ---------------------------------------------------------------
png("figs/residual_plots.png", width = 900, height = 450)
par(mfrow = c(1,2))
plot(fitted(fit), resid(fit), xlab = "Fitted values", ylab = "Residuals",
     main = "Residuals vs. Fitted", pch = 16, col = "darkgreen")
abline(h = 0, lty = 2)
qqnorm(resid(fit), main = "Normal Q-Q Plot of Residuals", pch = 16, col = "darkorange")
qqline(resid(fit), col = "red")
dev.off()

png("figs/resid_vs_age.png", width = 700, height = 500)
plot(cars$age, resid(fit), xlab = "Age (years)", ylab = "Residuals",
     main = "Residuals vs. Age", pch = 16, col = "purple")
abline(h = 0, lty = 2)
dev.off()

## ---------------------------------------------------------------
## 7. Largest residual: studentized residual, leverage, Cook's D
## ---------------------------------------------------------------
run_chunk("07_diagnostics", function() {
  fit <- readRDS("fit.rds")
  res <- resid(fit)
  idx <- which.max(abs(res))
  cat("Row with largest-magnitude residual:", idx, "\n")
  cat("That car's data: age =", cars$age[idx], ", miles =", cars$miles[idx],
      ", price =", cars$price[idx], "\n")
  cat("Raw residual =", res[idx], "\n\n")

  rstud <- rstudent(fit)
  hatv <- hatvalues(fit)
  cookd <- cooks.distance(fit)

  cat("Studentized residual for that car:", rstud[idx], "\n")
  cat("Leverage (hat value) for that car:", hatv[idx], "\n")
  cat("Cook's distance for that car:", cookd[idx], "\n\n")

  cat("All studentized residuals:\n"); print(round(rstud, 3))
  cat("\nAll leverages:\n"); print(round(hatv, 3))
  cat("\nAll Cook's distances:\n"); print(round(cookd, 3))

  n <- nrow(cars); p <- 2
  cat("\n--- Reference cutoffs ---\n")
  cat("Studentized residual: flag if |t| > 2 (rule of thumb)\n")
  cat("Leverage: flag if h_ii > 2p/n =", 2*p/n, "\n")
  cat("Cook's distance: flag if D > 4/n =", 4/n, " (or > 1 as a stronger rule)\n")
})

## ---------------------------------------------------------------
## 8. 90% CI for the slope
## ---------------------------------------------------------------
run_chunk("08_ci_slope", function() {
  s <- readRDS("stats.rds")
  fit <- readRDS("fit.rds")
  se_b1 <- summary(fit)$coefficients["age", "Std. Error"]
  b1 <- s$b1
  tcrit <- qt(0.95, df = s$n - 2)
  cat("b1 =", b1, "\n")
  cat("SE(b1) =", se_b1, "\n")
  cat("t* (df=", s$n-2, ", 90% CI) =", tcrit, "\n")
  ci <- b1 + c(-1, 1) * tcrit * se_b1
  cat("90% CI for slope: (", ci[1], ",", ci[2], ")\n")
  cat("\nConfirm with confint():\n")
  print(confint(fit, "age", level = 0.90))
})

## ---------------------------------------------------------------
## 9. r^2 two ways
## ---------------------------------------------------------------
run_chunk("09_r_squared", function() {
  s <- readRDS("stats.rds")
  r2_a <- s$r^2
  r2_b <- s$SSModel / s$SSY
  cat("r^2 (as square of correlation) =", r2_a, "\n")
  cat("r^2 (as SSModel/SSY)           =", r2_b, "\n")
  fit <- readRDS("fit.rds")
  cat("\nConfirm with summary(fit)$r.squared:\n")
  print(summary(fit)$r.squared)
})

## ---------------------------------------------------------------
## 10. Three tests for the linear relationship
## ---------------------------------------------------------------
run_chunk("10_tests", function() {
  cat("--- (a) Test for correlation ---\n")
  print(cor.test(cars$age, cars$price))

  cat("\n--- (b) Test for slope (t-test from lm summary) ---\n")
  fit <- readRDS("fit.rds")
  print(summary(fit)$coefficients)

  cat("\n--- (c) ANOVA F-test for regression ---\n")
  print(anova(fit))
})

## ---------------------------------------------------------------
## 11. Prediction at a chosen age
## ---------------------------------------------------------------
run_chunk("11_prediction", function() {
  fit <- readRDS("fit.rds")
  newage <- data.frame(age = 8)
  cat("Predicted value at age = 8:\n")
  print(predict(fit, newage))

  cat("\n90% CI for mean price at age = 8:\n")
  print(predict(fit, newage, interval = "confidence", level = 0.90))

  cat("\n90% PI for an individual car's price at age = 8:\n")
  print(predict(fit, newage, interval = "prediction", level = 0.90))
})

## ---------------------------------------------------------------
## 12. "Free car" age
## ---------------------------------------------------------------
run_chunk("12_free_car", function() {
  s <- readRDS("stats.rds")
  free_age <- -s$b0 / s$b1
  cat("b0 =", s$b0, "\n")
  cat("b1 =", s$b1, "\n")
  cat("Age at which predicted price = 0:  age = -b0/b1 =", free_age, "years\n")
})

cat("Done.\n")